 Experiment No.14

**Data Selection, Data Cleaning and Data Wrangling**

**1.Dataset Selection**

### Dataset Chosen: NLTK Movie Reviews Corpus

**Why this dataset?**
- It is a **standard benchmark** corpus widely used in NLP and Sentiment Analysis research.
- It is **publicly available** through NLTK (Python library) and catalogued by the Linguistic Data Consortium (LDC) under LDC2005T12.
- It contains **2000 movie reviews** labelled as *positive* or *negative*, making it ideal for a **binary text classification** ML problem.
- The data is **pre-structured** (files grouped by label folders), which simplifies initial loading.
- It represents **real-world text data** with noise, punctuation, and informal language — perfect for demonstrating data cleaning skills.

## 2. Environment Setup & Library Imports

In [5]:
import os
import re
import string
import warnings
warnings.filterwarnings('ignore')

# ─── Data Manipulation ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ─── NLP ──────────────────────────────────────────────────────────────────────
import nltk
from nltk.corpus import movie_reviews, stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer

# ─── Visualization ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

# ─── Download NLTK Resources ──────────────────────────────────────────────────
resources = ['movie_reviews', 'stopwords', 'punkt', 'wordnet', 'averaged_perceptron_tagger', 'punkt_tab']
for r in resources:
    nltk.download(r, quiet=True)

print(" All libraries imported successfully.")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   nltk    : {nltk.__version__}")

 All libraries imported successfully.
   pandas  : 2.2.2
   numpy   : 2.0.2
   nltk    : 3.9.1


**3.Load Dataset**

The dataset is loaded into a pandas dataframe for analysis.


The NLTK Movie Reviews corpus is loaded directly from the library. Each document belongs to one of two categories: **pos** (positive sentiment) or **neg** (negative sentiment).

In [6]:
# ─── Load raw documents from NLTK corpus ──────────────────────────────────────
documents = []
for category in movie_reviews.categories():
    for fileid in movie_reviews.fileids(category):
        raw_text = movie_reviews.raw(fileid)
        words     = movie_reviews.words(fileid)
        documents.append({
            'file_id'   : fileid,
            'category'  : category,           # 'pos' or 'neg'
            'raw_text'  : raw_text,
            'word_list' : list(words),
            'word_count': len(words)
        })

df_raw = pd.DataFrame(documents)
print(f"Total documents loaded : {len(df_raw)}")
print(f"Categories             : {df_raw['category'].unique()}")
print(f"\nShape of raw DataFrame : {df_raw.shape}")
df_raw.head(3)

Total documents loaded : 2000
Categories             : ['neg' 'pos']

Shape of raw DataFrame : (2000, 5)


,file_id,category,raw_text,word_list,word_count
0,neg/cv000_29416.txt,neg,"plot : two teen couples go to a church party ,...","[plot, :, two, teen, couples, go, to, a, churc...",879
1,neg/cv001_19502.txt,neg,the happy bastard's quick movie review \ndamn ...,"[the, happy, bastard, ', s, quick, movie, revi...",304
2,neg/cv002_17424.txt,neg,it is movies like these that make a jaded movi...,"[it, is, movies, like, these, that, make, a, j...",581


## 4. Attribute Description & Data Types

The table below documents every attribute in the dataset:

| # | Attribute | Data Type | Description | Role |
|---|-----------|-----------|-------------|------|
| 1 | `file_id` | `object` (str) | Unique filename from corpus (e.g. `pos/cv001_...txt`) | Identifier |
| 2 | `category` | `object` (str) | Sentiment label — `pos` or `neg` | Target variable |
| 3 | `raw_text` | `object` (str) | Full raw review text including punctuation, newlines | Input feature (raw) |
| 4 | `word_list` | `object` (list) | Tokenised list of words as provided by NLTK corpus | Intermediate |
| 5 | `word_count` | `int64` | Number of tokens per review | Numeric feature |

**Engineered attributes** (added during wrangling):

| # | Attribute | Data Type | Description |
|---|-----------|-----------|-------------|
| 6 | `label` | `int64` | Binary encoding: `pos`→1, `neg`→0 |
| 7 | `clean_text` | `object` (str) | Lowercased, punctuation-removed, stopword-filtered text |
| 8 | `clean_tokens` | `object` (list) | List of lemmatised, cleaned tokens |
| 9 | `char_count` | `int64` | Character length of clean_text |
| 10 | `sentence_count` | `int64` | Number of sentences in original review |
| 11 | `unique_word_count` | `int64` | Vocabulary richness — count of unique lemmas |
| 12 | `avg_word_length` | `float64` | Mean character length of tokens in clean_tokens |

In [7]:
# ─── Confirm dtypes ────────────────────────────────────────────────────────────
print("Data types in raw DataFrame:")
print(df_raw.dtypes)
print(f"\nMemory usage: {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB")

Data types in raw DataFrame:
file_id       object
category      object
raw_text      object
word_list     object
word_count     int64
dtype: object

Memory usage: 20455.3 KB


## 5. Data Quality Assessment

In [8]:
# ─── 5.1 Missing Values ────────────────────────────────────────────────────────
print("=" * 40)
print("Missing values per column:")
print(df_raw.isnull().sum())

# ─── 5.2 Duplicate Rows ────────────────────────────────────────────────────────
dup_count = df_raw.duplicated(subset=['file_id']).sum()
print(f"\nDuplicate file_id entries: {dup_count}")

# ─── 5.3 Class Distribution ───────────────────────────────────────────────────
print(f"\nClass distribution:")
print(df_raw['category'].value_counts())

# ─── 5.4 Word count statistics ────────────────────────────────────────────────
print(f"\nWord count statistics:")
print(df_raw['word_count'].describe().round(2))

Missing values per column:
file_id       0
category      0
raw_text      0
word_list     0
word_count    0
dtype: int64

Duplicate file_id entries: 0

Class distribution:
category
neg    1000
pos    1000
Name: count, dtype: int64

Word count statistics:
count    2000.00
mean      791.91
std       347.34
min        19.00
25%       560.00
50%       745.00
75%       957.25
max      2879.00
Name: word_count, dtype: float64


In [9]:
# ─── 5.5 Identify outliers by word count ──────────────────────────────────────
Q1 = df_raw['word_count'].quantile(0.25)
Q3 = df_raw['word_count'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_raw[(df_raw['word_count'] < lower_bound) | (df_raw['word_count'] > upper_bound)]
print(f"IQR range       : [{Q1:.0f}, {Q3:.0f}]")
print(f"Outlier bounds  : < {lower_bound:.0f} OR > {upper_bound:.0f}")
print(f"Outlier reviews : {len(outliers)} ({len(outliers)/len(df_raw)*100:.1f}%)")
print("\nNote: These outliers will NOT be dropped — extreme length is a valid text property.")

IQR range       : [560, 957]
Outlier bounds  : < -36 OR > 1553
Outlier reviews : 72 (3.6%)

Note: These outliers will NOT be dropped — extreme length is a valid text property.


## 6. Data Cleaning

### Cleaning Steps Applied:
1. **Lowercase** — normalize case variation
2. **Remove punctuation & special characters** — retain only alphabetic tokens
3. **Remove stopwords** — remove high-frequency words carrying little meaning (e.g., *the, is, and*)
4. **Lemmatization** — reduce words to base form (e.g., *running* → *run*)
5. **Remove very short tokens** (length < 2) — typically noise

### Attributes / Dimensions Filtered with Reasoning:
- `word_list` column **dropped** after cleaning — superseded by `clean_tokens`
- Reviews with `word_count < 50` **flagged** — extremely short reviews may be corrupt; however all are retained for transparency
- `raw_text` is **retained** alongside `clean_text` for reference

In [10]:
# ─── 6.1 Cleaning Function ─────────────────────────────────────────────────────
stop_words  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()

def clean_review(text: str) -> tuple:
    """
    Returns (clean_text: str, clean_tokens: list)
    """
    # Step 1: Lowercase
    text = text.lower()
    # Step 2: Remove punctuation and digits
    text = re.sub(r"[^a-z\s]", " ", text)
    # Step 3: Tokenise
    tokens = word_tokenize(text)
    # Step 4: Remove stopwords and short tokens, then lemmatise
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in stop_words and len(t) > 1
    ]
    clean_text = " ".join(tokens)
    return clean_text, tokens

print("Cleaning function defined. Running on all documents...")

Cleaning function defined. Running on all documents...


In [11]:
# ─── 6.2 Apply Cleaning ────────────────────────────────────────────────────────
cleaned = df_raw['raw_text'].apply(clean_review)
df_raw['clean_text']   = cleaned.apply(lambda x: x[0])
df_raw['clean_tokens'] = cleaned.apply(lambda x: x[1])

print("✅ Cleaning complete.")
print("\nSample — original vs cleaned:")
sample = df_raw.iloc[0]
print(f"  Original (first 200 chars): {sample['raw_text'][:200]}")
print(f"  Cleaned  (first 200 chars): {sample['clean_text'][:200]}")

✅ Cleaning complete.

Sample — original vs cleaned:
  Original (first 200 chars): plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . 
w
  Cleaned  (first 200 chars): plot two teen couple go church party drink drive get accident one guy dy girlfriend continues see life nightmare deal watch movie sorta find critique mind fuck movie teen generation touch cool idea pr


## 7. Data Wrangling — Feature Engineering

In [12]:
# ─── 7.1 Encode label ─────────────────────────────────────────────────────────
df_raw['label'] = df_raw['category'].map({'pos': 1, 'neg': 0})

# ─── 7.2 Derived numeric features ────────────────────────────────────────────
df_raw['char_count']        = df_raw['clean_text'].apply(len)
df_raw['sentence_count']    = df_raw['raw_text'].apply(lambda t: len(sent_tokenize(t)))
df_raw['unique_word_count'] = df_raw['clean_tokens'].apply(lambda t: len(set(t)))
df_raw['avg_word_length']   = df_raw['clean_tokens'].apply(
    lambda t: np.mean([len(w) for w in t]) if t else 0
).round(3)

print("Feature engineering complete. New columns added:")
print(df_raw[['label','char_count','sentence_count','unique_word_count','avg_word_length']].head())

Feature engineering complete. New columns added:
   label  char_count  sentence_count  unique_word_count  avg_word_length
0      0        2147              44                239            5.630
1      0         821              14                100            5.472
2      0        1771              25                197            5.763
3      0        1876              23                241            5.850
4      0        2766              38                287            6.059


In [13]:
# ─── 7.3 Drop intermediate / redundant columns ────────────────────────────────
# 'word_list' is superseded by clean_tokens
df_clean = df_raw.drop(columns=['word_list'])

# ─── 7.4 Reorder columns logically ───────────────────────────────────────────
col_order = [
    'file_id', 'category', 'label',
    'raw_text', 'clean_text', 'clean_tokens',
    'word_count', 'char_count', 'sentence_count',
    'unique_word_count', 'avg_word_length'
]
df_clean = df_clean[col_order]

print(f"Final cleaned DataFrame shape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")
df_clean.head(3)

Final cleaned DataFrame shape: (2000, 11)
Columns: ['file_id', 'category', 'label', 'raw_text', 'clean_text', 'clean_tokens', 'word_count', 'char_count', 'sentence_count', 'unique_word_count', 'avg_word_length']


,file_id,category,label,raw_text,clean_text,clean_tokens,word_count,char_count,sentence_count,unique_word_count,avg_word_length
0,neg/cv000_29416.txt,neg,0,"plot : two teen couples go to a church party ,...",plot two teen couple go church party drink dri...,"[plot, two, teen, couple, go, church, party, d...",879,2147,44,239,5.630
1,neg/cv001_19502.txt,neg,0,the happy bastard's quick movie review \ndamn ...,happy bastard quick movie review damn bug got ...,"[happy, bastard, quick, movie, review, damn, b...",304,821,14,100,5.472
2,neg/cv002_17424.txt,neg,0,it is movies like these that make a jaded movi...,movie like make jaded movie viewer thankful in...,"[movie, like, make, jaded, movie, viewer, than...",581,1771,25,197,5.763


In [14]:
# ─── 7.5 Final Data Types Summary ─────────────────────────────────────────────
print("Final DataFrame — dtype summary:")
dtype_summary = pd.DataFrame({
    'Column'      : df_clean.dtypes.index,
    'Dtype'       : df_clean.dtypes.values,
    'Non-Null'    : df_clean.notnull().sum().values,
    'Sample Value': [str(df_clean[c].iloc[0])[:60] for c in df_clean.columns]
})
print(dtype_summary.to_string(index=False))

Final DataFrame — dtype summary:
           Column   Dtype  Non-Null                                                 Sample Value
          file_id  object      2000                                          neg/cv000_29416.txt
         category  object      2000                                                          neg
            label   int64      2000                                                            0
         raw_text  object      2000 plot : two teen couples go to a church party , drink and the
       clean_text  object      2000 plot two teen couple go church party drink drive get acciden
     clean_tokens  object      2000 ['plot', 'two', 'teen', 'couple', 'go', 'church', 'party', '
       word_count   int64      2000                                                          879
       char_count   int64      2000                                                         2147
   sentence_count   int64      2000                                                           